In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import TimestampType
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import os

In [0]:
storage_account  = os.environ["STORAGE_ACCOUNT_NAME"]
storage_account_key = os.environ["STORAGE_ACCOUNT_KEY"]

bronze_gbfs_path  = f"abfss://bronze@{storage_account}.dfs.core.windows.net/gbfs_data/"
silver_status_path  = f"abfss://silver@{storage_account}.dfs.core.windows.net/gbfs_data/fact_station_status/"
checkpoint_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/gbfs_data/_checkpoints/gbfs_status/"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_account_key
)


## Configuring Auto Loader (cloudFiles)

In [0]:
print("Initializing Auto Loader Stream...")
raw_status_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    # Auto Loader will infer the schema and save it to the checkpoint path
    .option("cloudFiles.inferColumnTypes", "true") 
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("pathGlobFilter", "*_station_status.json")
    .load(bronze_gbfs_path)
)
print("complete")

## Defining the status information stream

In [0]:
status_cols_to_drop = [
    "num_scooters_available",
    "num_scooters_unavailable",
    "legacy_id",
    "eightd_has_available_keys",
    "is_installed",
    "num_bikes_disabled",
    "num_docks_disabled"
]

print("Applying transformations to the status stream...")

silver_status_stream = (
    raw_status_stream
    # 1. Flattening: Preserve the global feed timestamp, THEN explode the stations
    .select(
        F.col("last_updated").alias("feed_last_updated"), 
        F.explode("data.stations").alias("s")
    )
    .select("feed_last_updated", "s.*") 
    
    # 2. Hygiene: Drop the noisy commercial and hardware columns
    .drop(*status_cols_to_drop)
    
    # 3. Temporal Standardization: Convert Unix epochs to Timestamps
    .withColumn("last_reported_ts", F.from_unixtime(F.col("last_reported")).cast(TimestampType()))
    .withColumn("feed_timestamp", F.from_unixtime(F.col("feed_last_updated")).cast(TimestampType()))
    
    # Use the FEED timestamp for partitions to perfectly match your 6/23 to 6/26 files
    .withColumn("report_date", F.to_date(F.col("feed_timestamp")))
    .withColumn("report_hour", F.hour(F.col("feed_timestamp")))
    
    # 4. Logistical Flags: Injecting the business rules for stock-outs and saturation
    .withColumn(
        "is_stock_out", 
        F.when((F.col("num_bikes_available") == 0) & (F.col("is_renting") == 1), 1).otherwise(0)
    )
    .withColumn(
        "is_saturated", 
        F.when((F.col("num_docks_available") == 0) & (F.col("is_returning") == 1), 1).otherwise(0)
    )
)

print("Transformations successfully queued in the execution plan.")

## Executing the Write Stream

In [0]:
print("Initiating Write Stream to Silver Delta Table...")

status_write_query = (
    silver_status_stream.writeStream
    .format("delta")
    .outputMode("append") # High-velocity facts are always appended
    .option("checkpointLocation", checkpoint_path) # Fault tolerance
    .partitionBy("report_date", "report_hour") # Optimizes downstream querying
    # Triggering every 5 minutes to perfectly match your Azure Function's drop rate
    .trigger(processingTime="5 minutes") 
    .start(silver_status_path)
)

print("Stream is actively running and writing to Delta Lake!")

## Proceeing the station_dim

In [0]:
print("Processing station_information snapshots...")

# 1. Read all station_information JSONs from the raw Bronze path
raw_info_df = spark.read.json(f"{bronze_gbfs_path}*_station_information.json", multiLine=True)

# Define the columns mapped for removal during your local EDA
info_cols_to_drop = [
    "rental_uris",
    "rental_methods",
    "eightd_has_key_dispenser",
    "has_kiosk",
    "eightd_station_services",
    "electric_bike_surcharge_waiver"
]

# 2. Flatten the nested array and drop noisy commercial columns
# Notice we keep the top-level 'last_updated' timestamp to track the snapshot age
flat_info_df = (
    raw_info_df
    .select(F.col("last_updated"), F.explode("data.stations").alias("s"))
    .select("last_updated", "s.*")
    .drop(*info_cols_to_drop)
)

# 3. Deduplicate: Extract the absolute latest configuration per station
# This handles scenarios where Citi Bike updated a station's capacity or name during the day
window_spec = Window.partitionBy("station_id").orderBy(F.col("last_updated").desc())

latest_info_df = (
    flat_info_df
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1) # Keep only the newest record per station_id
    .drop("row_num", "last_updated") # Clean up temporary sorting columns
)

print(f"Isolated {latest_info_df.count()} unique active stations for the Silver Layer.")

In [0]:
# Define the path for the dim_stations silver table
silver_info_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/gbfs_data/dim_stations/"

print("Initiating Delta Lake Merge (SCD Type 1)...")

# Check if the Delta table already exists
if DeltaTable.isDeltaTable(spark, silver_info_path):
    # If it exists, connect to it and perform the Merge
    silver_table = DeltaTable.forPath(spark, silver_info_path)
    
    (silver_table.alias("target")
        .merge(
            latest_info_df.alias("source"),
            "target.station_id = source.station_id" # The Primary Key join
        )
        .whenMatchedUpdateAll()     # UPDATE if coordinates, name, or capacity changed
        .whenNotMatchedInsertAll()  # INSERT brand new street installations
        .execute()
    )
    print("Upsert complete. Dimension table synced with the latest physical reality.")

else:
    # First-run initialization: If the table doesn't exist, create it
    latest_info_df.write.format("delta").mode("overwrite").save(silver_info_path)
    print("Initial Silver dim_stations Delta table created successfully.")